# Day 09 · 멀티에이전트

어제 하네스의 구성요소(스킬 · 훅 · 플러그인)를 직접 만들었다. 오늘은 여기에 여러 에이전트로 구성된 팀을 추가한다.

> 어제 직접 만들어봤기 때문에, 오늘 메타 하네스가 생성한 파일을 열어서 이해할 수 있다.

| 랩 | 무엇을 | 배우는 것 |
|---|---|---|
| Lab 1 | **Superpowers** — 스트릭트 하네스를 그대로 써본다 | 강제가 주는 것 / 뺏는 것 |
| Lab 2 | **메타 하네스**로 멀티에이전트 팀을 생성한다 | 하네스를 만드는 하네스 |
| Lab 3 | 그 팀으로 **MCP 호스트 완성** | 위임하고 게이트로 검증 |

**어제의 두 산출물이 오늘 자란다**

```
harness/                       mcp-host/
├── skills/     ← 어제          ├── 채팅 UI     ← 어제
├── hooks/      ← 어제          ├── MCP 클라이언트  ← 오늘
├── skills/orchestrate/ ← 오늘  ├── 에이전트 루프   ← 오늘
└── agents/{planner,coder,      ├── 도구 승인 UI   ← 오늘
          reviewer,tester} ←오늘 └── 영속화(Neon)  ← 오늘
```

**판단 3문** — 멀티로 갈지 먼저 가린다.
① 병렬 분해가 가능한가? ② 가치가 **약 15배 토큰**보다 큰가? ③ 사람이 검수 가능한 단위인가?
**하나라도 No → 단일 + Plan Mode.**


## Lab 1 · Superpowers — 스트릭트 하네스 체험

다른 사람이 만든, 방법론이 코드로 강제되는 하네스를 그대로 사용해 본다.

### 1. 설치

```
/plugin marketplace add obra/superpowers-marketplace
/plugin install superpowers@superpowers-marketplace
```

### 2. 어제 앱에 기능 하나를 시킨다

```
/superpowers:brainstorm

"어제 만든 mcp-host 앱에 '대화 삭제' 기능을 추가하고 싶어"
```

그다음 흐름을 **그대로 따라간다**:

```
/superpowers:write-plan     # 구현 계획
/superpowers:execute-plan   # 서브에이전트 병렬 실행
```

### 3. 강제되는 항목 관찰

- [ ] **brainstorm이 먼저** 온다 — 바로 코드로 안 간다
- [ ] **실패하는 테스트를 먼저** 쓴다 (RED)
- [ ] 그다음 최소 구현으로 통과시킨다 (GREEN)
- [ ] **task마다 새 서브에이전트**가 뜬다 — 각자 깨끗한 컨텍스트
- [ ] 리뷰가 **2단계**로 돈다 (스펙 준수 → 코드 품질)

### 4. 우회를 시도한다 ★

```
테스트 건너뛰고 바로 구현해줘
```

**막히는지 확인한다.** 그게 **Hard Gate**다 — 부탁이 아니라 강제.

> 어제 만든 `guard.mjs` 훅과 같은 원리다. 다만 **막는 대상이 파일이 아니라 절차**다.

### 5. 판정 — 표를 채운다

| | 강제가 준 것 | 강제가 뺏은 것 |
|---|---|---|
| 품질 | | |
| 속도 | | |
| 자유도 | | |

### 원리

**정적(스트릭트) 하네스 = 미리 정해진 팀 구조.** 내가 팀을 설계하지 않아도 프레임워크가 강제한다.
우회가 안 되는 게 **장점이자 단점**이다 — 내 도메인에 안 맞아도 **못 바꾼다**.

> 다음 랩에서는 우리 도메인에 맞는 팀을 직접 생성한다.


## Lab 2 · 메타 하네스로 멀티에이전트 팀 구축

Superpowers는 다른 사람이 정한 방법론이다. 이번에는 우리 도메인에 맞는 팀을 직접 구성한다.

### 1. 메타 하네스 설치

```
# 병렬 서브에이전트 활성화 (필수 — 없으면 팀이 뜨지 않는다)
export CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1

# Claude Code
/plugin marketplace add revfactory/harness
/plugin install harness@harness-marketplace

# Codex 사용자 — 포트: github.com/SaehwanPark/meta-harness
```

> **왜 이걸 켜야 하나** — orchestrate 스킬에 "병렬로 위임하라"고 적어도, 그것만으로는 강제되지 않는다(스킬은 부탁). revfactory/harness는 Claude Code의 **네이티브 agent team 기능**(TeamCreate 등)으로 실제 병렬 서브에이전트를 띄우는데, 이 기능은 위 환경변수로 켜야 동작한다.

> `harness`는 **하네스를 만드는 하네스**다 — 도메인 한 줄을 주면 **에이전트 팀 + 그들이 쓸 스킬**을 생성한다.
> Codex 쪽 포트는 `meta-harness`.

### 2. 팀 생성

```
/harness "MCP 호스트 웹앱 개발팀 — Next.js · Prisma · MCP · 에이전트 루프.
          기획 → 병렬 구현 → 읽기전용 리뷰 → 테스트 게이트 순서를 강제한다."
```

생성되는 것:

```
.claude/
├── agents/
│   ├── planner.md     기획·분해
│   ├── coder.md       TDD 구현
│   ├── reviewer.md    읽기전용 감사
│   └── tester.md      검증 게이트
└── skills/
    └── orchestrate/SKILL.md   리드 — 위임만 한다
```

### 3. 생성된 파일 열어보기 ★

```bash
cat .claude/agents/reviewer.md
cat .claude/skills/orchestrate/SKILL.md
```

**확인할 것:**

| 항목 | 왜 보나 |
|---|---|
| `name` · `description` | description이 곧 **트리거**다 |
| **`tools`** | **이게 권한 경계다** — reviewer에 `Write`가 있으면 안 된다 |
| 역할 프롬프트 | "직접 코딩하지 말고 위임만" 이 들어있나 |
| 순서 강제 | 건너뛰기 금지가 명시됐나 |

> 어제 `SKILL.md`를 직접 작성해봤기 때문에, 이 파일이 특별한 게 아니라 프론트매터가 붙은 마크다운이라는 걸 알 수 있다.

### 4. 손본다 — 권한을 좁힌다

```yaml
# .claude/agents/reviewer.md
tools: Read, Grep, Glob     # Write · Bash 제거 — 못 고치게
model: haiku                # 리뷰는 싼 모델로 (모델 라우팅)
```

```yaml
# .claude/agents/coder.md
tools: Read, Write, Edit, Bash
# 역할 프롬프트에: "자기 worktree 안에서만 쓴다"
```

**tools가 곧 권한 경계다.** 프롬프트로 부탁하는 것과 달리, 권한 제한은 우회할 수 없다.

### 5. 어제 만든 훅을 붙인다

```bash
cp harness/hooks/guard.mjs .claude/hooks/
```

이제 **누가 시키든** `.env`는 못 건드린다 — @coder가 쓰기 권한이 있어도.

### 6. 검증

```bash
claude plugin validate ./harness
```

`/agents` 로 팀이 뜨는지 확인한다.

### 원리

**메타 하네스는 필요한 팀을 그때그때 생성해서 쓰는 방식이다.**

| | 정적 (Superpowers) | 메타 (harness) |
|---|---|---|
| 팀 구조 | **정해져 있다** | **직접 생성한다** |
| 도메인 적합 | 범용 | **내 도메인에 맞춘다** |
| 수정 | 어렵다 | **파일을 열어 고친다** |
| 전제 | — | **내부 구조를 알아야 수정할 수 있다** (어제 실습) |


## Lab 3 · 멀티에이전트로 기능 추가 (핵심)

전날 만든 챗앱(Next.js · Neon · Qwen 스트리밍)에, Lab 1·2에서 만든 하네스 팀으로 **여러 기능을 병렬 추가**한다.
목표는 기능 자체가 아니라 **분해** — 잘 나눠지는 것과 안 나눠지는 것을 직접 확인한다.

### 기능 백로그

| 기능 | 파일(신규) |
|---|---|
| 웹검색 도구 | `lib/tools/web-search.ts` + `api/tools/search/route.ts` |
| 구글 OAuth | `lib/auth.ts` + `api/auth/.../route.ts` + 버튼 |
| MCP 서버 모듈 장착 | `lib/mcp/*` + `api/mcp/servers/*` + `components/mcp/mcp-panel.tsx` |
| 대화 사이드바 | `api/conversations/*` + `components/chat/conversation-sidebar.tsx` |
| 도구 로그 뷰어 | `components/chat/tool-log.tsx` |
| 모델·프롬프트 설정 | `components/chat/settings-panel.tsx` |
| 테마 토글 | `components/theme/*` |
| 내보내기(md) | `lib/export/to-markdown.ts` + 버튼 |
| 토큰·비용 배지 | `components/chat/token-badge.tsx` |

### Step 0 · agent teams 켜기

```bash
export CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1
```
없으면 병렬 서브에이전트가 뜨지 않는다.

### Step 1 · 계약 먼저 (0단계)

```
/harness:orchestrate "위 백로그 9개 기능을 추가한다"

@planner 가 먼저 SHARED CONTRACT 를 확정:
  - 공유 타입: Tool · SearchResult · McpServer · ToolCallLog · SessionUser
  - SSE 이벤트: token | tool_call | usage | done
  - prisma/schema.prisma 를 단일 마이그레이션으로 한 번에
→ 사람이 계획 승인
```

> Prisma 스키마는 단일 파일·단일 마이그레이션이라, 모델 변경을 여기서 전부 못박아야 이후가 병렬이 된다.

### Step 2 · 병렬 구현 (배치 1)

계약이 서면 @coder 9명이 각자 worktree에서 신규 파일을 **동시에** 구현한다. 대부분 신규 파일이라 서로 안 겹친다.

```
Batch 1 (동시): 웹검색 · MCP 코어 · OAuth · 사이드바 · 로그뷰어
                · 설정 · 테마 · 토큰배지 · 내보내기
```

### Step 3 · 의존 처리 (배치 2)

```
Batch 2 (선행 대기):
  MCP 관리 UI      ← MCP API 계약 확정 후
  채팅 루프 통합    ← 웹검색·MCP 도구 등록 + 설정 저장 필드 후
```

### 병렬이 막히는 지점 (직접 관찰)

| 파일 / 자원 | 왜 병렬 불가 | 대응 |
|---|---|---|
| `prisma/schema.prisma` | 단일 마이그레이션 | 계약 단계에서 한 번에 |
| `api/chat/route.ts` | 도구·이벤트·설정 병합점 | 단독 소유, 순차 |
| `lib/auth.ts` · 세션 | OAuth가 공유 상태 변경 | 세션 shape 계약 고정 → 다른 기능은 `user.id`만 읽어 병렬 유지 |
| `[id]/route.ts` | 사이드바·설정 공유 | 소유권 지정 or 필드 계약 선합의 |
| `app/layout.tsx` | 컴포넌트 마운트 병합 | 셸 단독 소유, 컴포넌트는 병렬 |

> **OAuth는 세션 때문에 그냥은 병렬이 안 되지만, 계약을 먼저 고정하면 격리해서 동시 진행할 수 있다.** 무엇이 왜 안 나뉘는지 아는 것이 오케스트레이션의 핵심이다 (판단 3문 ①).

### Step 4 · 리뷰어 검증

@reviewer(읽기 전용)가 이음새를 점검한다: 웹검색과 MCP가 같은 `Tool` 모양인가, 로그 뷰어·토큰 배지가 계약의 SSE 이벤트 이름을 쓰는가, 세션 필드가 일치하는가. 계약을 지켰으면 병합, 어긋났으면 담당 chunk로 되돌린다.

### 무엇을 봤나

- 9개 기능이 대부분 **신규 파일**이라 병렬로 갈렸다 — 멀티에이전트가 실제로 도는 그림.
- `schema.prisma` · `chat/route.ts` · `auth.ts` 는 공유 자원이라 **병렬을 막았다** — 이게 분해의 경계다.
- 계약을 먼저 고정한 덕에 병렬 결과가 조합됐다.


## Lab 3-부록 · 핵심 기능 붙이는 스텝

백로그 대부분은 하네스가 코드를 짜면 끝나지만, **구글 OAuth·웹검색·MCP 모듈**은 외부 설정(콘솔·API 키)이 먼저다. 이 부분은 사람이 하고, 나머지 코드는 하네스에 맡긴다.

---

### 구글 OAuth 붙이기

**① Google Cloud Console**
- [console.cloud.google.com](https://console.cloud.google.com) → 프로젝트 생성
- API 및 서비스 → **OAuth 동의 화면** (External, 앱 이름·지원 이메일)
- **사용자 인증 정보** → OAuth 클라이언트 ID → **웹 애플리케이션**
- **승인된 리디렉션 URI** (정확히):
  ```
  http://localhost:3000/api/auth/callback/google
  https://<프로덕션도메인>/api/auth/callback/google
  ```

**② `.env`**
```bash
AUTH_GOOGLE_ID=...
AUTH_GOOGLE_SECRET=...
AUTH_SECRET=          # npx auth secret 로 생성
```

**③ Auth.js v5 설정**
```ts
// auth.ts
import NextAuth from "next-auth"
import Google from "next-auth/providers/google"
import { PrismaAdapter } from "@auth/prisma-adapter"

export const { handlers, auth, signIn, signOut } = NextAuth({
  adapter: PrismaAdapter(prisma),
  providers: [Google],   // AUTH_GOOGLE_ID / AUTH_GOOGLE_SECRET 자동 인식
})

// app/api/auth/[...nextauth]/route.ts
export { GET, POST } from "@/auth"
```

**④ 로그인 버튼**
```tsx
"use client"
import { signIn } from "next-auth/react"
<button onClick={() => signIn("google")}>Google로 로그인</button>
```

> **가장 흔한 오류 — `redirect_uri_mismatch`**: 콘솔의 승인된 리디렉션 URI가 `/api/auth/callback/google` 와 프로토콜·포트·경로까지 정확히 일치해야 한다.

---

### 웹검색 도구 붙이기

**① 검색 API 키** — [tavily.com](https://tavily.com) 가입 → API 키 (무료 1,000/월, 카드 불필요). AI 에이전트용이라 스니펫이 깔끔하다. (키 없이 keyless 데모도 가능.)

**② 설치 + env**
```bash
npm i @tavily/core
# .env
TAVILY_API_KEY=tvly-...
```

**③ 도구 구현 (공유 계약의 Tool 모양, 서버 전용)**
```ts
// lib/tools/web-search.ts
import { tavily } from "@tavily/core"
const tvly = tavily({ apiKey: process.env.TAVILY_API_KEY })

export const webSearch: Tool = {
  name: "web_search",
  description: "웹에서 최신 정보를 검색한다",
  parameters: { query: "string" },
  async execute({ query }) {
    const r = await tvly.search(query, { maxResults: 5 })
    return r.results.map(x => ({ title: x.title, url: x.url, snippet: x.content }))
  },
}
```

**④ 에이전트 루프에 등록** → LLM이 필요할 때 `web_search` 를 호출.

> 검색 키도 **API Route(서버)에서만** 쓴다. 프론트에 두면 유출된 키다 (8강 원칙).

---

### MCP 서버 모듈 장착

`claude mcp add` 를 웹앱으로 재현한다.

**① `McpServer` 모델** (name·transport·url|command·enabled) 을 Neon에 저장.
**② `/api/mcp/servers`** — 추가·삭제·토글.
**③ 연결 시 도구 등록:**
```ts
// lib/mcp/registry.ts (요지)
async function connect(server: McpServer) {
  const client = await mcpConnect(server)     // stdio | http
  const tools = await client.listTools()
  for (const t of tools) registerTool(toTool(t))  // McpTool → Tool 어댑트
}
const tools = getRegisteredTools()   // 채팅 루프는 목록만 읽는다
```

> **노출한 도구가 곧 경계**: 서버를 꽂으면 그 도구만 에이전트에 노출되고, `enabled` 토글이 곧 권한 스위치다. 5강에서 만든 MCP 서버도 여기에 붙는다.


## Lab 2.5 · 오케스트레이터 하네스 정형화 (검증된 프롬프트)

Lab 1·2에서 만든 하네스를 **재현 가능한 절차**로 정리한다. 아래 프롬프트는 실제로 웹개발 과제(태스크 진행 보드)에 돌려 다듬은 것이다.

### 구축 5단계

1. **규약** — `CLAUDE.md`에 스택·컨벤션·금지사항 (모든 에이전트의 공유 기준)
2. **리드** — 위임만 하는 `orchestrate` 스킬. 순서는 @planner → @coder 병렬 → @reviewer
3. **역할 에이전트** — planner·coder·reviewer·tester. `tools`로 권한 경계
4. **강제 게이트** — `CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1` + guard 훅
5. **검증** — @reviewer가 이음새(공유 계약)를 점검

### @planner 프롬프트 (분해 전용)

```
코드를 쓰지 말고 분해만 한다. 도구 불필요.

규약(인라인): <스택·컨벤션을 여기 붙인다>
목표: <한 문장>
분해 규칙:
 · 독립 병렬 구현 가능한 파일 단위 3~4개
 · 각 파일 자기완결 (공유 타입은 각 파일 정의)
 · 공유 계약(상태 enum·Task 타입)을 먼저 고정한다   ← 중요
출력 형식(고정):
 CHUNK n · 파일 · 내용 · 인수조건 · 의존성
```

> **실행에서 확인한 것** — 규약을 붙이지 않고 "알아서 분해하라"고만 하면 플래너가 파일도 안 읽고 엉뚱한 결과를 냈다. 규약 인라인 + 출력 형식 고정이 재현성을 만든다.

### @coder 프롬프트 (파일 하나 전용)

```
배정된 파일 하나만 구현한다. 다른 파일 금지.

규약(인라인): <스택·컨벤션>
배정 chunk: 파일 · 내용 · 인수조건
절차:
 1. 파일을 쓴다 (Write)
 2. 다시 읽어 인수조건을 스스로 점검
 3. 보고: 인수조건 충족 여부 + 경로
```

### @reviewer 프롬프트 (읽기 전용)

```
읽기 전용. 수정·실행 금지. 읽고 판정만.

검수 항목:
 1. 각 파일이 규약을 지키나
 2. 파일 간 정합성 — 상태 어휘·타입이 파일마다 일치하나  ← 핵심
 3. 인수조건 충족 여부
보고: 파일별 판정 + 발견한 정합성 문제(심각도) + 종합(병합/수정)
```

### 실행에서 나온 핵심 교훈 — 병렬의 이음새

태스크 보드 기능을 4명이 병렬 구현한 결과, 개별 파일은 규약을 다 지켰지만 **공유 계약이 어긋났다**:

| 상태 | board | badge | card | feed |
|---|---|---|---|---|
| 진행 | `in-progress` | `running` | `in-progress` | `running` |
| 대기 | `waiting` | `waiting` | `waiting` | `queued` |
| 제목 필드 | `title` | — | `title` | `label` |

같은 "진행" 상태가 `in-progress`와 `running`으로 갈려, 합치면 타입 에러가 난다.
"각 파일 자기완결"로 파일 충돌은 막았지만, 상태 enum·타입 같은 **공유 계약이 어긋난 것**이다.

**대응** — 플래너가 공유 계약(상태 enum, Task 타입)을 계획에 **먼저 못박고**, @reviewer 게이트가 이를 검증한다. 리뷰어가 없으면 조합 단계에서 터진다. 이것이 병렬 오케스트레이션에서 리뷰어가 선택이 아니라 필수인 이유다.


### 개선 — 계약을 먼저 고정하는 플래너

플래너 프롬프트에 **SHARED CONTRACT를 먼저 확정**하는 단계 하나를 추가한다.

```
분해 규칙 (개선판):
1. 먼저 공유 계약(SHARED CONTRACT)을 확정한다:
   - TaskStatus 리터럴 유니온을 하나로 (정확한 문자열)
   - Task 타입의 필드 이름·타입
   - 상태별 Tailwind 색 계열
2. 그다음 파일로 분해. 각 파일은 자기완결이되,
   위 계약을 그대로 재정의해서 쓴다 (계약은 공유, 코드는 각자 복제).

출력:
SHARED CONTRACT
- TaskStatus: 'pending' | 'running' | 'done' | 'failed'
- Task: { id, title, status, updatedAt }
- 색: pending=slate · running=blue · done=green · failed=red
CHUNK n · 파일 · 내용 · 계약 준수 · 인수조건 · 의존성
```

@coder 에게도 이 SHARED CONTRACT를 함께 전달한다.

### 전후 비교 (실측)

같은 과제를 **프롬프트만 바꿔** 병렬 재구현한 결과:

| | 느슨한 프롬프트 | 계약 고정 프롬프트 |
|---|---|---|
| 진행 상태 | `in-progress` / `running` 혼재 | 모두 `running` |
| 대기 상태 | `waiting` / `queued` 혼재 | 모두 `pending` |
| 제목 필드 | `title` / `label` 혼재 | 모두 `title` |
| 합쳤을 때 | **타입 에러** | 정합 (에러 없음) |

바꾼 것은 **플래너 프롬프트에 계약 정의 단계 하나를 더한 것**뿐이다.
공유 계약을 계획 단계에서 못박으면 병렬 구현의 이음새가 사라진다.


## 산출물 체크리스트

**기성 하네스 사용** (Lab 1)
- [ ] Superpowers를 Claude Code 또는 Codex에 설치해 강제 흐름을 겪었다
- [ ] 우회를 시도했고 Hard Gate에 막혔다

**메타 하네스로 팀 생성** (Lab 2)
- [ ] 메타 하네스로 도메인 팀을 생성하고 `agents/*.md` 를 열어 확인했다
- [ ] 어제 만든 `guard.mjs` 훅을 붙였다

**하네스 정형화** (Lab 2.5)
- [ ] 검증된 planner·coder·reviewer 프롬프트를 확인했다
- [ ] 계약을 먼저 고정하면 이음새가 사라지는 것을 봤다

**멀티에이전트로 기능 추가** (Lab 3)
- [ ] `CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1` 을 켜고 시작했다
- [ ] @planner가 **공유 계약 + Prisma 스키마**를 먼저 확정했다
- [ ] 계획을 **사람이 승인**한 뒤 병렬 구현이 시작됐다
- [ ] @coder들이 각자 worktree에서 신규 파일을 **동시에** 구현했다
- [ ] **웹검색 · 구글 OAuth · MCP 서버 모듈**이 붙었다
- [ ] **병렬이 막히는 지점**(schema · chat route · auth · layout)을 직접 확인했다
- [ ] @reviewer가 이음새(Tool 모양 · 이벤트 이름 · 세션 필드)를 검증했다

> 핵심은 기능이 아니라 분해였다. 계약을 먼저 고정하고, 병렬 가능한 것과 아닌 것을 갈랐다.
